# LangChain LangGraph 工作流学习Demo

本Notebook将深入学习LangGraph - LangChain的状态图框架。

## 什么是LangGraph?

LangGraph是一个用于构建**有状态的、多Actor应用程序**的框架。它扩展了LangChain，专注于：
- 🔄 **循环和分支**: 支持复杂的控制流
- 💾 **状态管理**: 跨步骤持久化状态
- 🎯 **人机交互**: 支持人工审核和干预
- ⚡ **并行执行**: 支持并行节点执行

## 学习大纲
1. **LangGraph基础** - 图、节点、边、状态
2. **条件路由** - 动态决策和分支
3. **循环和迭代** - 实现复杂的迭代逻辑
4. **人机协作** - 人工审核节点
5. **实战案例** - 构建复杂的Agent工作流

## 环境准备

In [ ]:
# 安装依赖
# pip install langchain==0.3.15
# pip install langchain-core==0.3.28
# pip install langchain-community==0.3.14
# pip install langgraph==0.2.64
# pip install dashscope==1.20.11

## 导入库并配置API Key

In [ ]:
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 验证API Key
if DASHSCOPE_API_KEY:
    print("✅ API Key已加载")
else:
    print("❌ 请在.env文件中配置DASHSCOPE_API_KEY")

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 初始化LLM
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.7,
    dashscope_api_key=DASHSCOPE_API_KEY
)

print("✅ LLM初始化完成")

---

# 第一部分：LangGraph基础

## 1.1 核心概念

LangGraph的核心组件：
- **State (状态)**: 在节点间传递的数据
- **Nodes (节点)**: 执行具体任务的函数
- **Edges (边)**: 连接节点，定义执行流程
- **Graph (图)**: 由节点和边组成的工作流

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# 定义状态
class AgentState(TypedDict):
    """Agent的状态定义"""
    messages: Annotated[list, add_messages]  # 消息列表，自动追加新消息
    user_input: str  # 用户输入
    current_step: str  # 当前步骤
    result: str  # 结果

print("✅ 状态定义完成")

## 1.2 创建简单的线性工作流

In [ ]:
# 定义节点函数
def input_node(state: AgentState) -> AgentState:
    """输入处理节点"""
    print(f"📥 [输入节点] 接收用户输入: {state['user_input']}")
    state["current_step"] = "input_processed"
    return state

def process_node(state: AgentState) -> AgentState:
    """处理节点"""
    print(f"⚙️  [处理节点] 处理中...")
    # 使用LLM处理输入
    from langchain_core.messages import HumanMessage
    
    response = llm.invoke([
        HumanMessage(content=f"请简短回答: {state['user_input']}")
    ])
    
    state["result"] = response.content
    state["current_step"] = "processed"
    print(f"✅ [处理节点] 处理完成")
    return state

def output_node(state: AgentState) -> AgentState:
    """输出节点"""
    print(f"📤 [输出节点] 输出结果: {state['result'][:50]}...")
    state["current_step"] = "completed"
    return state

print("✅ 节点函数定义完成")

In [ ]:
# 构建工作流图
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("input", input_node)
workflow.add_node("process", process_node)
workflow.add_node("output", output_node)

# 添加边（定义执行顺序）
workflow.set_entry_point("input")  # 设置起始节点
workflow.add_edge("input", "process")  # input -> process
workflow.add_edge("process", "output")  # process -> output
workflow.add_edge("output", END)  # output -> END

# 编译图
app = workflow.compile()

print("✅ 工作流图构建完成")

In [ ]:
# 测试工作流
print("\n" + "="*60)
print("🚀 开始执行工作流")
print("="*60 + "\n")

initial_state = {
    "messages": [],
    "user_input": "什么是LangGraph?",
    "current_step": "start",
    "result": ""
}

final_state = app.invoke(initial_state)

print("\n" + "="*60)
print("✅ 工作流执行完成")
print("="*60)
print(f"\n最终结果: {final_state['result']}")

---

# 第二部分：条件路由

条件路由允许根据状态动态选择下一个节点。

## 2.1 简单的条件分支

In [ ]:
from typing import Literal

# 扩展状态定义
class ConditionalState(TypedDict):
    """带条件的状态"""
    question: str
    question_type: str  # "math", "general", "code"
    answer: str

# 分类节点
def classify_question(state: ConditionalState) -> ConditionalState:
    """分类问题类型"""
    question = state["question"].lower()
    
    if any(word in question for word in ["计算", "加", "减", "乘", "除", "数学"]):
        state["question_type"] = "math"
        print(f"🔢 问题分类: 数学问题")
    elif any(word in question for word in ["代码", "编程", "python", "函数"]):
        state["question_type"] = "code"
        print(f"💻 问题分类: 编程问题")
    else:
        state["question_type"] = "general"
        print(f"💡 问题分类: 一般问题")
    
    return state

# 不同类型的处理节点
def handle_math(state: ConditionalState) -> ConditionalState:
    """处理数学问题"""
    print(f"🔢 [数学处理器] 处理数学问题...")
    response = llm.invoke(f"作为数学专家，请回答: {state['question']}")
    state["answer"] = f"[数学] {response.content}"
    return state

def handle_code(state: ConditionalState) -> ConditionalState:
    """处理编程问题"""
    print(f"💻 [代码处理器] 处理编程问题...")
    response = llm.invoke(f"作为编程专家，请回答: {state['question']}")
    state["answer"] = f"[编程] {response.content}"
    return state

def handle_general(state: ConditionalState) -> ConditionalState:
    """处理一般问题"""
    print(f"💡 [通用处理器] 处理一般问题...")
    response = llm.invoke(f"请回答: {state['question']}")
    state["answer"] = f"[通用] {response.content}"
    return state

print("✅ 条件节点定义完成")

In [ ]:
# 路由函数
def route_question(state: ConditionalState) -> Literal["math", "code", "general"]:
    """根据问题类型路由到不同的处理器"""
    return state["question_type"]

# 构建条件工作流
conditional_workflow = StateGraph(ConditionalState)

# 添加节点
conditional_workflow.add_node("classify", classify_question)
conditional_workflow.add_node("math", handle_math)
conditional_workflow.add_node("code", handle_code)
conditional_workflow.add_node("general", handle_general)

# 设置入口
conditional_workflow.set_entry_point("classify")

# 添加条件边
conditional_workflow.add_conditional_edges(
    "classify",  # 从classify节点
    route_question,  # 使用route_question函数决定下一步
    {
        "math": "math",
        "code": "code",
        "general": "general"
    }
)

# 所有处理器都连接到END
conditional_workflow.add_edge("math", END)
conditional_workflow.add_edge("code", END)
conditional_workflow.add_edge("general", END)

# 编译
conditional_app = conditional_workflow.compile()

print("✅ 条件工作流构建完成")

In [ ]:
# 测试不同类型的问题
test_questions = [
    "计算 25 * 4 是多少?",
    "如何用Python实现快速排序?",
    "什么是人工智能?"
]

for question in test_questions:
    print("\n" + "="*60)
    print(f"❓ 问题: {question}")
    print("="*60)
    
    result = conditional_app.invoke({
        "question": question,
        "question_type": "",
        "answer": ""
    })
    
    print(f"\n💬 回答: {result['answer'][:100]}...\n")

---

# 第三部分：循环和迭代

LangGraph支持循环，可以实现迭代改进、自我反思等模式。

## 3.1 自我反思循环

In [ ]:
class ReflectionState(TypedDict):
    """反思状态"""
    task: str
    draft: str
    critique: str
    iteration: int
    max_iterations: int
    is_good: bool

def generate_draft(state: ReflectionState) -> ReflectionState:
    """生成初稿"""
    print(f"\n✍️  [生成器] 生成第 {state['iteration']} 版草稿...")
    
    if state["iteration"] == 1:
        prompt = f"请为以下任务创建一个解决方案: {state['task']}"
    else:
        prompt = f"""任务: {state['task']}

之前的版本: {state['draft']}

改进建议: {state['critique']}

请基于改进建议，生成改进后的版本。"""
    
    response = llm.invoke(prompt)
    state["draft"] = response.content
    print(f"✅ 草稿生成完成 (长度: {len(state['draft'])} 字符)")
    return state

def reflect(state: ReflectionState) -> ReflectionState:
    """反思和评审"""
    print(f"\n🤔 [评审器] 评审草稿...")
    
    prompt = f"""请评审以下方案，并给出改进建议。

任务: {state['task']}
方案: {state['draft']}

如果方案已经很好，请说"方案良好，无需改进"。
否则，请给出具体的改进建议。
"""
    
    response = llm.invoke(prompt)
    state["critique"] = response.content
    
    # 检查是否满意
    state["is_good"] = "无需改进" in state["critique"] or "方案良好" in state["critique"]
    
    if state["is_good"]:
        print(f"✅ 方案已达标")
    else:
        print(f"💭 需要改进: {state['critique'][:100]}...")
    
    state["iteration"] += 1
    return state

def should_continue(state: ReflectionState) -> Literal["generate", "end"]:
    """决定是否继续迭代"""
    if state["is_good"] or state["iteration"] > state["max_iterations"]:
        return "end"
    return "generate"

print("✅ 反思节点定义完成")

In [ ]:
# 构建反思工作流
reflection_workflow = StateGraph(ReflectionState)

# 添加节点
reflection_workflow.add_node("generate", generate_draft)
reflection_workflow.add_node("reflect", reflect)

# 设置入口
reflection_workflow.set_entry_point("generate")

# 添加边
reflection_workflow.add_edge("generate", "reflect")

# 添加条件边（循环或结束）
reflection_workflow.add_conditional_edges(
    "reflect",
    should_continue,
    {
        "generate": "generate",  # 继续循环
        "end": END  # 结束
    }
)

# 编译
reflection_app = reflection_workflow.compile()

print("✅ 反思工作流构建完成")

In [ ]:
# 测试反思循环
print("\n" + "="*60)
print("🔄 开始自我反思循环")
print("="*60)

result = reflection_app.invoke({
    "task": "设计一个简单的任务管理系统",
    "draft": "",
    "critique": "",
    "iteration": 1,
    "max_iterations": 3,
    "is_good": False
})

print("\n" + "="*60)
print("✅ 反思循环完成")
print("="*60)
print(f"\n总迭代次数: {result['iteration'] - 1}")
print(f"\n最终方案:\n{result['draft']}")

---

# 第四部分：人机协作

LangGraph支持人工介入，可以在关键节点暂停等待人工审核。

## 4.1 带人工审核的工作流

In [ ]:
class HumanInLoopState(TypedDict):
    """人机协作状态"""
    request: str
    ai_proposal: str
    human_feedback: str
    approved: bool
    final_result: str

def ai_propose(state: HumanInLoopState) -> HumanInLoopState:
    """AI提出方案"""
    print(f"\n🤖 [AI] 生成方案...")
    
    response = llm.invoke(f"请为以下需求提出解决方案: {state['request']}")
    state["ai_proposal"] = response.content
    
    print(f"✅ AI方案: {state['ai_proposal'][:100]}...")
    return state

def human_review(state: HumanInLoopState) -> HumanInLoopState:
    """人工审核（模拟）"""
    print(f"\n👤 [人工审核] 等待审核...")
    
    # 实际应用中，这里应该暂停等待真实的人工输入
    # 这里我们模拟人工反馈
    
    # 模拟审核逻辑
    if len(state["ai_proposal"]) > 50:  # 简单的审核标准
        state["approved"] = True
        state["human_feedback"] = "方案详细，批准执行"
        print(f"✅ 审核通过")
    else:
        state["approved"] = False
        state["human_feedback"] = "方案过于简单，请提供更详细的方案"
        print(f"❌ 审核未通过，需要修改")
    
    return state

def finalize(state: HumanInLoopState) -> HumanInLoopState:
    """最终确定"""
    print(f"\n📋 [最终化] 生成最终结果...")
    
    state["final_result"] = f"""最终方案（已审核）:

{state['ai_proposal']}

审核意见: {state['human_feedback']}
"""
    print(f"✅ 最终结果已生成")
    return state

def check_approval(state: HumanInLoopState) -> Literal["approved", "rejected"]:
    """检查是否通过审核"""
    return "approved" if state["approved"] else "rejected"

print("✅ 人机协作节点定义完成")

In [ ]:
# 构建人机协作工作流
human_loop_workflow = StateGraph(HumanInLoopState)

# 添加节点
human_loop_workflow.add_node("propose", ai_propose)
human_loop_workflow.add_node("review", human_review)
human_loop_workflow.add_node("finalize", finalize)

# 设置入口
human_loop_workflow.set_entry_point("propose")

# 添加边
human_loop_workflow.add_edge("propose", "review")

# 添加条件边
human_loop_workflow.add_conditional_edges(
    "review",
    check_approval,
    {
        "approved": "finalize",
        "rejected": "propose"  # 如果未通过，重新生成方案
    }
)

human_loop_workflow.add_edge("finalize", END)

# 编译
human_loop_app = human_loop_workflow.compile()

print("✅ 人机协作工作流构建完成")

In [ ]:
# 测试人机协作
print("\n" + "="*60)
print("🤝 开始人机协作流程")
print("="*60)

result = human_loop_app.invoke({
    "request": "设计一个用户登录系统",
    "ai_proposal": "",
    "human_feedback": "",
    "approved": False,
    "final_result": ""
})

print("\n" + "="*60)
print("✅ 人机协作完成")
print("="*60)
print(f"\n{result['final_result']}")

---

# 第五部分：实战案例 - 智能客服系统

综合运用前面学到的知识，构建一个完整的智能客服工作流。

In [ ]:
class CustomerServiceState(TypedDict):
    """客服系统状态"""
    user_query: str
    intent: str  # "question", "complaint", "request"
    urgency: str  # "high", "medium", "low"
    response: str
    escalate: bool  # 是否升级到人工
    satisfaction_score: int  # 满意度评分 (1-5)

def analyze_intent(state: CustomerServiceState) -> CustomerServiceState:
    """分析用户意图"""
    print(f"\n🔍 [意图分析] 分析用户查询...")
    
    prompt = f"""分析以下用户查询的意图和紧急程度。

用户查询: {state['user_query']}

请以JSON格式返回:
{{
  "intent": "question/complaint/request",
  "urgency": "high/medium/low"
}}
"""
    
    response = llm.invoke(prompt).content
    
    # 简化处理，实际应该解析JSON
    if "complaint" in response.lower():
        state["intent"] = "complaint"
    elif "request" in response.lower():
        state["intent"] = "request"
    else:
        state["intent"] = "question"
    
    if "high" in response.lower():
        state["urgency"] = "high"
    elif "low" in response.lower():
        state["urgency"] = "low"
    else:
        state["urgency"] = "medium"
    
    print(f"✅ 意图: {state['intent']}, 紧急度: {state['urgency']}")
    return state

def handle_question(state: CustomerServiceState) -> CustomerServiceState:
    """处理问题"""
    print(f"\n❓ [问题处理] 回答用户问题...")
    response = llm.invoke(f"作为客服，请专业地回答: {state['user_query']}")
    state["response"] = response.content
    state["escalate"] = False
    return state

def handle_complaint(state: CustomerServiceState) -> CustomerServiceState:
    """处理投诉"""
    print(f"\n😔 [投诉处理] 处理用户投诉...")
    
    if state["urgency"] == "high":
        state["response"] = "非常抱歉给您带来不便。此问题已升级到专员处理，我们会在1小时内联系您。"
        state["escalate"] = True
        print(f"⚠️  高优先级投诉，升级到人工")
    else:
        response = llm.invoke(f"作为客服，请诚恳地处理以下投诉: {state['user_query']}")
        state["response"] = response.content
        state["escalate"] = False
    
    return state

def handle_request(state: CustomerServiceState) -> CustomerServiceState:
    """处理请求"""
    print(f"\n📋 [请求处理] 处理用户请求...")
    response = llm.invoke(f"作为客服，请协助处理以下请求: {state['user_query']}")
    state["response"] = response.content
    state["escalate"] = False
    return state

def evaluate_satisfaction(state: CustomerServiceState) -> CustomerServiceState:
    """评估满意度"""
    print(f"\n⭐ [满意度评估] 评估服务质量...")
    
    # 模拟满意度评分
    if state["escalate"]:
        state["satisfaction_score"] = 3
    elif state["intent"] == "complaint":
        state["satisfaction_score"] = 4
    else:
        state["satisfaction_score"] = 5
    
    print(f"评分: {state['satisfaction_score']}/5")
    return state

def route_by_intent(state: CustomerServiceState) -> str:
    """根据意图路由"""
    return state["intent"]

print("✅ 客服系统节点定义完成")

In [ ]:
# 构建客服工作流
customer_service_workflow = StateGraph(CustomerServiceState)

# 添加节点
customer_service_workflow.add_node("analyze", analyze_intent)
customer_service_workflow.add_node("handle_question", handle_question)
customer_service_workflow.add_node("handle_complaint", handle_complaint)
customer_service_workflow.add_node("handle_request", handle_request)
customer_service_workflow.add_node("evaluate", evaluate_satisfaction)

# 设置入口
customer_service_workflow.set_entry_point("analyze")

# 添加条件路由
customer_service_workflow.add_conditional_edges(
    "analyze",
    route_by_intent,
    {
        "question": "handle_question",
        "complaint": "handle_complaint",
        "request": "handle_request"
    }
)

# 所有处理节点都连接到评估
customer_service_workflow.add_edge("handle_question", "evaluate")
customer_service_workflow.add_edge("handle_complaint", "evaluate")
customer_service_workflow.add_edge("handle_request", "evaluate")
customer_service_workflow.add_edge("evaluate", END)

# 编译
customer_service_app = customer_service_workflow.compile()

print("✅ 客服工作流构建完成")

In [ ]:
# 测试客服系统
test_queries = [
    "你们的产品保修期是多久？",
    "我的订单已经延迟3天了，非常不满意！",
    "我想申请退货"
]

for query in test_queries:
    print("\n" + "="*60)
    print(f"👤 用户: {query}")
    print("="*60)
    
    result = customer_service_app.invoke({
        "user_query": query,
        "intent": "",
        "urgency": "",
        "response": "",
        "escalate": False,
        "satisfaction_score": 0
    })
    
    print(f"\n🤖 客服: {result['response']}")
    print(f"\n📊 统计:")
    print(f"   意图: {result['intent']}")
    print(f"   紧急度: {result['urgency']}")
    print(f"   升级人工: {'是' if result['escalate'] else '否'}")
    print(f"   满意度: {result['satisfaction_score']}/5")
    print()

---

# 总结

## 学到的内容

### 1. LangGraph基础
- ✅ **状态管理**: TypedDict定义状态结构
- ✅ **节点函数**: 执行具体任务的函数
- ✅ **边和流程**: 定义执行顺序
- ✅ **图的编译**: 将定义转换为可执行的应用

### 2. 条件路由
- ✅ **条件边**: 根据状态动态选择下一个节点
- ✅ **路由函数**: 实现复杂的决策逻辑
- ✅ **多分支处理**: 支持多个处理路径

### 3. 循环和迭代
- ✅ **自我反思**: 迭代改进输出质量
- ✅ **循环控制**: 设置最大迭代次数
- ✅ **退出条件**: 动态决定何时结束循环

### 4. 人机协作
- ✅ **人工审核节点**: 在关键点暂停等待人工输入
- ✅ **反馈机制**: 整合人工反馈到工作流
- ✅ **升级策略**: 自动判断何时需要人工介入

### 5. 实战应用
- ✅ **智能客服**: 综合运用多种技术
- ✅ **意图识别**: 动态路由到不同处理器
- ✅ **质量评估**: 自动评估服务质量

## LangGraph vs 传统Agent

| 特性 | 传统Agent | LangGraph |
|------|-----------|----------|
| 控制流 | 线性/简单分支 | 复杂图结构 |
| 状态管理 | 有限 | 完整的状态系统 |
| 循环支持 | 困难 | 原生支持 |
| 人工介入 | 不支持 | 原生支持 |
| 可视化 | 困难 | 容易（图结构） |

## 最佳实践

1. **状态设计**
   - 使用TypedDict清晰定义状态
   - 只在状态中保存必要信息
   - 使用Annotated处理列表累加

2. **节点设计**
   - 节点职责单一明确
   - 总是返回更新后的状态
   - 添加日志便于调试

3. **流程设计**
   - 绘制流程图再实现
   - 设置合理的退出条件
   - 处理异常情况

4. **性能优化**
   - 避免不必要的LLM调用
   - 使用缓存减少重复计算
   - 考虑异步执行

## 下一步学习

- 探索LangGraph的持久化功能
- 学习并行节点执行
- 研究子图和模块化设计
- 实现生产级的错误处理
- 学习LangGraph Studio可视化工具